<a href="https://colab.research.google.com/github/Sandesh-Pandey/GenAI_ML_DL_NLP/blob/main/SentimentAnalysisRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install datasets 'tensorflow==2.21.0'


In [11]:
import zipfile
import os


In [12]:
zip_path = '/content/Ex_Files_Foundation_Math_Generative_AI.zip'
extract_dir = '/content/content/Ex_Files_Foundation_Math_Generative_AI.zip/extracted_files'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [13]:
!pip install gensim
!pip install -U datasets huggingface_hub gensim
import tensorflow as tf
from tensorflow.keras.layers import Input, TextVectorization, GRU, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from gensim.models import Word2Vec
from datasets import load_dataset
import numpy as np

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached huggingface_hub-1.24.0-py3-none-any.whl.metadata (16 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached huggingface_hub-1.24.0-py3-none-any.whl (771 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.19.4
    Uninstalling huggingface-hub-0.19.4:
      Successfully uninstalled huggingface-hub-0.19.4
  Attempting uninstall: datasets
    Found existing installation: datasets 2.16.1
    Uninstalling datasets-2.16.1:
      Successfully uninstalled datasets-2.16.1


In [1]:
# Consolidate and install specific, compatible versions of libraries
!pip install 'tensorflow==2.21.0' datasets==2.16.1 huggingface_hub==0.19.4 gensim


  Using cached datasets-2.16.1-py3-none-any.whl.metadata (20 kB)
  Using cached huggingface_hub-0.19.4-py3-none-any.whl.metadata (14 kB)
Using cached datasets-2.16.1-py3-none-any.whl (507 kB)
Using cached huggingface_hub-0.19.4-py3-none-any.whl (311 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.24.0
    Uninstalling huggingface_hub-1.24.0:
      Successfully uninstalled huggingface_hub-1.24.0
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.6.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.19.4 which is incompat

In [7]:
import tensorflow as tf
from datasets import load_dataset
# Enable CuDNN for GRU is available
if tf.config.list_physical_devices('GPU'):
  GRU_LAYER = lambda units, return_sequences=False: tf.keras.layers.GRU(
      units, return_sequences=return_sequences, recurrent_activation='sigmoid')
else:
    GRU_LAYER = lambda units, return_sequences=False: tf.keras.layers.GRU(
      units, return_sequences=return_sequences,dropout=0.15,recurrent_dropout=0.5)

#Load the IMDB dataset from Hugging Face
imdb_dataset = load_dataset('imdb')
print(imdb_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [8]:
from datasets import load_dataset
from gensim.models import Word2Vec
from tensorflow.keras.layers import TextVectorization
import tensorflow as tf
import numpy as np

dataset = load_dataset("imdb")

max_vocab_size = 25000
max_seq_len = 50
embedding_dim = 150

train_text = dataset["train"]["text"]

sentences = [text.split() for text in train_text]

word2vec_model = Word2Vec(
    sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=1,
    workers=4
)

vectorizer = TextVectorization(
    max_tokens=max_vocab_size,
    output_mode="int",
    output_sequence_length=max_seq_len
)

vectorizer.adapt(tf.data.Dataset.from_tensor_slices(train_text).batch(256))

vocab = vectorizer.get_vocabulary()
word_index = {word: idx for idx, word in enumerate(vocab)}

embedding_matrix = np.zeros((len(vocab), embedding_dim))

for word, idx in word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[idx] = word2vec_model.wv[word]

In [9]:
# Create TensorFlow datasets
train_texts = tf.convert_to_tensor(dataset['train']['text'])
train_labels = tf.convert_to_tensor(dataset['train']['label'])
test_texts = tf.convert_to_tensor(dataset['test']['text'])
test_labels = tf.convert_to_tensor(dataset['test']['label'])

# Vectorize and preprocess text data
def preprocess_texts(text, label):
    text = vectorizer(text)
    label = tf.one_hot(label, depth=2)
    return text, label

train_ds = tf.data.Dataset.from_tensor_slices((train_texts, train_labels))
train_ds = train_ds.map(preprocess_texts).shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_texts, test_labels))
test_ds = test_ds.map(preprocess_texts).batch(256).prefetch(tf.data.AUTOTUNE)


In [12]:

# Encoder-only architecture
class SentimentClassifier(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, latent_dim, output_dim, embedding_matrix):
        super(SentimentClassifier, self).__init__()
        self.embedding = tf.keras.layers.Embedding(
            vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True)
        self.dropout = tf.keras.layers.Dropout(0.25)
        self.gru = GRU_LAYER(latent_dim, return_sequences=False)
        self.dense = tf.keras.layers.Dense(output_dim, activation="softmax")

    def build(self, input_shape):
        # Explicitly build sub-layers to ensure parameters are initialized
        # input_shape is (None, max_seq_len) from sentiment_model.build(input_shape=(None, max_seq_len))
        self.embedding.build(input_shape)
        embedding_output_shape = self.embedding.compute_output_shape(input_shape)

        self.dropout.build(embedding_output_shape)
        dropout_output_shape = self.dropout.compute_output_shape(embedding_output_shape)

        self.gru.build(dropout_output_shape)
        gru_output_shape = self.gru.compute_output_shape(dropout_output_shape)

        self.dense.build(gru_output_shape)
        super(SentimentClassifier, self).build(input_shape)

    def call(self, inputs):
        x = self.embedding(inputs)
        x = self.dropout(x)
        x = self.gru(x)
        outputs = self.dense(x)
        return outputs

# Instantiate the model
sentiment_model = SentimentClassifier(len(vocab), embedding_dim, latent_dim, output_dim, embedding_matrix)

# Build the model
sentiment_model.build(input_shape=(None, max_seq_len))

# Compile the model
sentiment_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
sentiment_model.summary()

Model: "sentiment_classifier_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 50, 150)        │     3,750,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 512)            │     1,019,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,770,930 (18.20 MB)

 Trainable params: 1,020,930 (3.89 MB)

 Non-trainable params: 3,750,000 (14.31 MB)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
sentiment_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    batch_size=256,
    callbacks=[early_stopping]
)

Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 318s 3s/step - accuracy: 0.7076 - loss: 0.6251 - val_accuracy: 0.5772 - val_loss: 0.7616
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 322s 3s/step - accuracy: 0.7197 - loss: 0.5590 - val_accuracy: 0.5831 - val_loss: 0.6789
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 331s 3s/step - accuracy: 0.7282 - loss: 0.5933 - val_accuracy: 0.6476 - val_loss: 0.6269
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 314s 3s/step - accuracy: 0.7544 - loss: 0.5445 - val_accuracy: 0.6558 - val_loss: 0.6217
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 330s 3s/step - accuracy: 0.7625 - loss: 0.5293 - val_accuracy: 0.6554 - val_loss: 0.6277
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 313s 3s/step - accuracy: 0.7787 - loss: 0.5071 - val_accuracy: 0.6797 - val_loss: 0.5995
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 321s 3s/step - accuracy: 0.7801 - loss: 0.4862 - val_accuracy: 0.6920 - val_loss: 0.5817
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 313s 3s/step - accuracy: 0.7836 - loss: 0.4800 - val_accuracy: 0.6945 - v

In [15]:

# Manual testing
sample_positive = "This was the best movie I have ever seen."
sample_negative = "This was the worst movie I have ever watched."
sample_neutral = "The movie was okay, not great but not terrible."
sample_sarcasm = "Wow, this was such a masterpiece... the actors, the screenplay, I could stay for hours if it wasn't for how bad it was."
sample_irony = "The plot was so riveting, I couldn’t stop yawning."

# Preprocess the samples
sample_positive_vectorized = vectorizer(tf.convert_to_tensor([sample_positive]))
sample_negative_vectorized = vectorizer(tf.convert_to_tensor([sample_negative]))
sample_neutral_vectorized = vectorizer(tf.convert_to_tensor([sample_neutral]))
sample_sarcasm_vectorized = vectorizer(tf.convert_to_tensor([sample_sarcasm]))
sample_irony_vectorized = vectorizer(tf.convert_to_tensor([sample_irony]))

# Predict sentiment
positive_prediction = sentiment_model.predict(sample_positive_vectorized)
negative_prediction = sentiment_model.predict(sample_negative_vectorized)
neutral_prediction = sentiment_model.predict(sample_neutral_vectorized)
sarcasm_prediction = sentiment_model.predict(sample_sarcasm_vectorized)
irony_prediction = sentiment_model.predict(sample_irony_vectorized)

print("Positive Prediction:", positive_prediction)
print("Negative Prediction:", negative_prediction)
print("Neutral Prediction:", neutral_prediction)
print("Sarcasm Prediction:", sarcasm_prediction)
print("Irony Prediction:", irony_prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 493ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Positive Prediction: [[0.03019157 0.9698084 ]]
Negative Prediction: [[0.98289454 0.01710546]]
Neutral Prediction: [[0.68756974 0.3124302 ]]
Sarcasm Prediction: [[0.744122   0.25587803]]
Irony Prediction: [[0.5638105  0.43618938]]
